# DS 2002 Midterm Project — Walmart Hurricane Analytics

**Team analysis notebook | Fall 2026**

This project analyzes Walmart transactions alongside historical weather observations to investigate demand changes, product-name inconsistencies, differences among stores, and possible data-quality artifacts. We compare **September 1–10, 2024** with **September 11–20, 2024** as the analysis windows used in the team's progress work. These dates are **not independently verified as hurricane impact dates**.

**Reproducibility:** Run all cells in order. Place `walmart_transactions.csv` (or `walmart_transactions (1).csv`) in the repository `data/` folder, notebook directory, or Colab working directory. Weather analysis requires internet access to Open-Meteo.

## 1. Load and prepare transaction data

In [ ]:
import pandas as pd

from pathlib import Path

candidates = [
    Path("data/walmart_transactions.csv"),
    Path("data/walmart_transactions (1).csv"),
    Path("../data/walmart_transactions.csv"),
    Path("../data/walmart_transactions (1).csv"),
    Path("walmart_transactions.csv"),
    Path("walmart_transactions (1).csv"),
    Path("/content/walmart_transactions.csv"),
    Path("/content/walmart_transactions (1).csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Walmart CSV not found. Add it to the repository data/ folder "
        "or upload it to Colab, then rerun."
    )
print("Loading:", data_path)
df = pd.read_csv(data_path)

print("Rows loaded:", len(df))

df = df.drop_duplicates()

df["quantity"] = pd.to_numeric(
    df["quantity"], errors="coerce"
)

df = df[df["quantity"] > 0].copy()

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="mixed",
    errors="coerce"
)

df = df.dropna(subset=["timestamp"])
df["date"] = df["timestamp"].dt.normalize()

daily_orders = (
    df.groupby("date")["transaction_id"]
    .nunique()
    .sort_index()
)

print("Data date range:", daily_orders.index.min(),
      "to", daily_orders.index.max())

baseline_start = "2024-09-01"
baseline_end = "2024-09-10"

surge_start = "2024-09-11"
surge_end = "2024-09-20"

baseline_data = daily_orders.loc[
    baseline_start:baseline_end
]

surge_data = daily_orders.loc[
    surge_start:surge_end
]

if baseline_data.empty or surge_data.empty:
    raise ValueError(
        "One or both windows contain no data. "
        "Check the printed dataset date range."
    )

dates = pd.date_range(
    baseline_start, surge_end, freq="D"
)

daily_orders = daily_orders.reindex(
    dates, fill_value=0
)

baseline_avg = daily_orders.loc[
    baseline_start:baseline_end
].mean()

surge_avg = daily_orders.loc[
    surge_start:surge_end
].mean()

multiple = (
    surge_avg / baseline_avg
    if baseline_avg > 0 else float("nan")
)

print("\nQ1 — Demand Surge")
print("Baseline window:", baseline_start, "to", baseline_end)
print("Surge window:", surge_start, "to", surge_end)
print("Baseline average:", round(baseline_avg, 2), "orders/day")
print("Surge average:", round(surge_avg, 2), "orders/day")
print("Demand multiple:", round(multiple, 2), "x")


## 2. Q1 — Demand surge
Compare average daily unique transactions in the baseline and surge windows.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.figure(figsize=(12, 5))

plt.plot(
    daily_orders.index,
    daily_orders.values,
    marker="o",
    linewidth=2,
    label="Daily Orders"
)

plt.axvspan(
    pd.Timestamp(baseline_start),
    pd.Timestamp(baseline_end) + pd.Timedelta(days=1),
    color="lightblue",
    alpha=0.4,
    label="Baseline Period"
)

plt.axvspan(
    pd.Timestamp(surge_start),
    pd.Timestamp(surge_end) + pd.Timedelta(days=1),
    color="lightcoral",
    alpha=0.3,
    label="Surge Period"
)

plt.axhline(
    baseline_avg,
    color="blue",
    linestyle="--",
    label=f"Baseline Avg: {baseline_avg:.1f}"
)

plt.axhline(
    surge_avg,
    color="red",
    linestyle="--",
    label=f"Surge Avg: {surge_avg:.1f}"
)

plt.title("Walmart Demand Surge: Daily Order Volume")
plt.xlabel("Date")
plt.ylabel("Number of Orders")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()


**Finding:** Walmart's average daily orders went from 575.2 during September 1–10 to 679.1 during September 11–20, showing that demand increased by about 18%, or 1.18 times.

## 3. Q2 — SKU consolidation
Normalize product names and verify that total quantities are preserved.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

sku_df = df.copy()

sku_col = "sku"
product_col = "product_name"

original_skus = sku_df[sku_col].nunique()

def canonicalize(name):
    name = str(name).lower().strip()
    name = re.sub(r"[^a-z0-9]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    if "strawberry" in name and (
        "pop tart" in name or "poptart" in name
    ):
        return "pop tarts strawberry"

    if "monopoly" in name:
        return "monopoly board game"

    if "water" in name and "24" in name:
        return "bottled water 24 pack"

    if "aa" in name and (
        "batteries" in name or "battery" in name
    ):
        return "aa batteries 4 pack"

    return name

sku_df["canonical_product"] = (
    sku_df[product_col].apply(canonicalize)
)

canonical_count = sku_df["canonical_product"].nunique()

units_before = sku_df["quantity"].sum()

consolidated = (
    sku_df.groupby("canonical_product", as_index=False)
    ["quantity"].sum()
)

units_after = consolidated["quantity"].sum()

assert units_before == units_after, \
    "ERROR: Unit totals changed!"

families = (
    sku_df.groupby("canonical_product")
    .agg(
        distinct_skus=(sku_col, "nunique"),
        total_units=("quantity", "sum")
    )
    .sort_values(
        ["distinct_skus", "total_units"],
        ascending=False
    )
)

largest_family = families.index[0]

print("Q2 — SKU Consolidation")
print("Original distinct SKUs:", original_skus)
print("Canonical products:", canonical_count)
print("Largest consolidated family:", largest_family)
print("SKUs in largest family:",
      families.iloc[0]["distinct_skus"])
print("Units before consolidation:", units_before)
print("Units after consolidation:", units_after)
print("Assertion passed: Unit totals unchanged!")

comparison = pd.DataFrame({
    "Category": ["Original SKUs", "Canonical Products"],
    "Count": [original_skus, canonical_count]
})

display(comparison)
display(families.head(10))

plt.figure(figsize=(7, 4))

bars = plt.bar(
    comparison["Category"],
    comparison["Count"],
    color=["steelblue", "seagreen"]
)

plt.title("SKU Consolidation: Before vs. After")
plt.ylabel("Number of Distinct Products")
plt.bar_label(bars)
plt.tight_layout()
plt.show()


**Q2 answer:** We started with 45 different SKUs and consolidated them into 42 canonical products by combining different names for the same items. Strawberry Pop-Tarts was the largest consolidated family with 4 SKUs and 5,005 units. The total number of units stayed the same at 47,897, showing that no units were lost during consolidation.

## 4. Q3 — Weather API and daily-order correlation
Join historical weather to daily order counts by date, validate the merge, and calculate correlations. **Limitation:** the weather coordinates (Charlottesville, Virginia) are not matched to the Florida store locations, so the correlation should not be interpreted as local hurricane exposure.

In [ ]:
import pandas as pd
import requests
import matplotlib.pyplot as plt

orders = df.copy()
orders["date"] = pd.to_datetime(
    orders["timestamp"], format="mixed", errors="coerce"
).dt.normalize()

daily = (
    orders.dropna(subset=["date"])
    .groupby("date")["transaction_id"]
    .nunique()
    .reset_index(name="daily_orders")
)

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 38.03,
    "longitude": -78.51,
    "start_date": "2024-09-01",
    "end_date": "2024-09-30",
    "daily": "precipitation_sum,temperature_2m_max",
    "timezone": "America/New_York"
}

source = "Live API"

try:
    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    weather_json = response.json()

    weather = pd.DataFrame({
        "date": pd.to_datetime(weather_json["daily"]["time"]),
        "precipitation_mm": weather_json["daily"]["precipitation_sum"],
        "max_temp_c": weather_json["daily"]["temperature_2m_max"]
    })

except Exception as e:
    raise RuntimeError(
        f"Weather API request failed: {e}. "
        "No fallback data was substituted."
    )

assert weather["date"].is_unique, "Duplicate weather dates!"

rows_before = len(daily)

merged = daily.merge(
    weather,
    on="date",
    how="left",
    validate="one_to_one",
    indicator=True
)

rows_after = len(merged)

assert rows_before == rows_after, "Rows were lost!"
assert (merged["_merge"] == "both").all(), \
    "Some dates did not match weather records!"

merged = merged.drop(columns="_merge")

valid_rain = merged.dropna(
    subset=["daily_orders", "precipitation_mm"]
)

valid_temp = merged.dropna(
    subset=["daily_orders", "max_temp_c"]
)

rain_n = len(valid_rain)
temp_n = len(valid_temp)

rain_corr = (
    valid_rain["daily_orders"].corr(
        valid_rain["precipitation_mm"]
    )
    if rain_n >= 3 else float("nan")
)

temp_corr = (
    valid_temp["daily_orders"].corr(
        valid_temp["max_temp_c"]
    )
    if temp_n >= 3 else float("nan")
)

print("Q3 — Weather Correlation")
print("Weather source:", source)
print("API URL:", response.url)
print("Daily rows before join:", rows_before)
print("Daily rows after join:", rows_after)
print("Join validation: Passed")
print("Rainfall correlation:", round(rain_corr, 3))
print("Rainfall sample size:", rain_n, "days")
print("Temperature correlation:", round(temp_corr, 3))
print("Temperature sample size:", temp_n, "days")

if rain_n < 30 or temp_n < 30:
    print("Limitation: Small sample size.")

display(merged.head(10))

plt.figure(figsize=(7, 5))
plt.scatter(
    valid_rain["precipitation_mm"],
    valid_rain["daily_orders"]
)
plt.title(
    f"Rainfall vs. Walmart Orders (r = {rain_corr:.2f})"
)
plt.xlabel("Daily Precipitation (mm)")
plt.ylabel("Daily Orders")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Q3 answer, with its caveat:** We used live historical weather data from the Open-Meteo API and matched it with Walmart's daily orders. The row count stayed at 30 before and after the join, so no days were lost. We found a moderate positive correlation between rainfall and orders (0.445) and a negative correlation between temperature and orders (-0.33). This suggests that orders tended to increase on rainier and cooler days, but since we only looked at 30 days of data, we can't draw strong conclusions.

## 5. Q4 — Store-level demand comparison
Clean inconsistent store IDs before comparing average daily orders. The source does not include store cities or coordinates, so this is a **store-level**, not a mapped geographic, comparison.

In [ ]:
# Normalize all observed store-ID variants before grouping.
store_df = df.copy()
store_df["store_id"] = (
    store_df["store_id"].astype(str).str.strip().str.upper()
    .str.replace("_", "-", regex=False)
    .str.replace(r"^FL(?=\d)", "FL-", regex=True)
)

baseline = store_df[store_df["date"].between(baseline_start, baseline_end)]
surge = store_df[store_df["date"].between(surge_start, surge_end)]

baseline_days = (pd.Timestamp(baseline_end) - pd.Timestamp(baseline_start)).days + 1
surge_days = (pd.Timestamp(surge_end) - pd.Timestamp(surge_start)).days + 1

store_results = pd.concat([
    baseline.groupby("store_id")["transaction_id"].nunique().rename("baseline_orders"),
    surge.groupby("store_id")["transaction_id"].nunique().rename("surge_orders")
], axis=1).fillna(0)
store_results["baseline_daily"] = store_results["baseline_orders"] / baseline_days
store_results["surge_daily"] = store_results["surge_orders"] / surge_days
store_results["percent_change"] = 100 * (
    store_results["surge_daily"] / store_results["baseline_daily"].replace(0, float("nan")) - 1
)
store_results = store_results.sort_values("percent_change", ascending=False)
print("Standardized stores:", len(store_results))
display(store_results.round(2))

plt.figure(figsize=(11, 5))
store_results["percent_change"].plot(kind="bar")
plt.title("Change in average daily orders by standardized store")
plt.xlabel("Store ID")
plt.ylabel("Change (%)")
plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Q4 — Stores and Geography

We compared Walmart transaction activity across 12 stores during the baseline period (September 1–10, 2024) and the surge period (September 11–20, 2024). We cleaned the store IDs by standardizing capitalization, removing extra spaces, and correcting inconsistent hyphens and underscores. We then calculated the percentage change in average daily orders for each store.

**Store FL-330 experienced the largest increase in demand, rising from 453 to 590 orders, a 30.24% increase.** This was followed by FL-078, which increased from 471 to 569 orders (20.81%), and FL-287, which increased from 470 to 565 orders (20.21%). Store FL-710 experienced the smallest increase at 10.73%.

All 12 stores experienced an increase in demand during the selected periods, although the magnitude varied across locations. These findings suggest that Walmart should prioritize additional inventory and staffing at stores experiencing the largest increases in demand during severe weather events. However, because the dataset does not provide specific city or county locations, further geographic analysis would require additional store location information.

## 6. Q5 — Signal or artifact?
Check duplicates, missingness, unique transactions, and whether transaction IDs span stores or dates. These checks assess some data-quality risks; they do not establish causation.

In [ ]:

# Q5 — Signal or Artifact Analysis

import pandas as pd

# Make a copy of the original data
q5_df = df.copy()

# Standardize store IDs
q5_df["store_id"] = (
    q5_df["store_id"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace("_", "-", regex=False)
    .str.replace(r"^FL(?=\d)", "FL-", regex=True)
)

# Check for exact duplicate rows
duplicate_rows = q5_df.duplicated().sum()

# Check for missing values
missing_values = q5_df.isnull().sum()

# Count unique transactions
unique_transactions = q5_df["transaction_id"].nunique()

# Compare baseline and surge periods
baseline = q5_df[
    q5_df["date"].between(baseline_start, baseline_end)
]

surge = q5_df[
    q5_df["date"].between(surge_start, surge_end)
]

baseline_orders = baseline["transaction_id"].nunique()
surge_orders = surge["transaction_id"].nunique()

baseline_daily = baseline_orders / (
    (pd.Timestamp(baseline_end) -
     pd.Timestamp(baseline_start)).days + 1
)

surge_daily = surge_orders / (
    (pd.Timestamp(surge_end) -
     pd.Timestamp(surge_start)).days + 1
)

change = (
    (surge_daily - baseline_daily)
    / baseline_daily
) * 100

print("Q5 — Signal or Artifact")
print("Exact duplicate rows:", duplicate_rows)
print("Unique transactions:", unique_transactions)
print("\nMissing values:")
print(missing_values)
print("\nBaseline daily orders:", round(baseline_daily, 2))
print("Surge daily orders:", round(surge_daily, 2))
print("Percentage change:", round(change, 2), "%")


In [ ]:

# Additional transaction consistency checks

transaction_checks = q5_df.groupby("transaction_id").agg(
    store_count=("store_id", "nunique"),
    date_count=("date", "nunique")
)

print(
    "Transactions assigned to multiple stores:",
    (transaction_checks["store_count"] > 1).sum()
)

print(
    "Transactions assigned to multiple dates:",
    (transaction_checks["date_count"] > 1).sum()
)


### Q5 — Signal or Artifact

We analyzed the Walmart transaction dataset to determine whether the increase in customer demand during the selected hurricane-related periods was a real pattern or an artifact caused by data-quality issues.

Our analysis identified 24,521 unique transactions, with no exact duplicate rows or missing values. We also standardized inconsistent store IDs to prevent the same stores from being treated as separate locations.

The results showed that average daily orders increased from 575.2 during the baseline period to 679.1 during the surge period, representing an 18.06% increase. This increase remained evident when counting unique transaction IDs rather than individual product rows.

Additional consistency checks confirmed that zero transactions were associated with multiple stores or multiple dates. These findings strengthen the evidence that the observed increase is not simply a result of duplicate records or inconsistent transaction assignments.

Overall, the results suggest a genuine increase in recorded Walmart shopping activity during the selected period. Although the analysis does not establish that the hurricane directly caused the increase, Walmart could use these findings to support inventory planning and staffing decisions during future severe weather events.

## 7. Recommendations and limitations

- **Prepare for higher demand:** Recorded average daily orders rose from **575.2** to **679.1** (approximately **18.06%**) between the two selected windows. Use this as a planning scenario, not a proven hurricane effect.
- **Prioritize store-level response:** After standardizing store IDs, **FL-330** showed the largest reported increase (**30.24%**). Consider flexible staffing and inventory buffers for higher-growth stores, while validating the finding against the final run's table.
- **Improve master data:** Standardize store identifiers and product names upstream to avoid fragmented reporting. The SKU analysis reduced **45 SKUs to 42 canonical product groups** while preserving unit totals.
- **Improve weather matching:** Replace the Charlottesville weather point with geographically appropriate observations for each store and align analysis windows to documented storm timing before making hurricane-specific claims.
- **Avoid causal overstatement:** The 30-day weather correlation and transaction-quality checks describe associations and consistency, not a controlled causal effect.

## 8. Team reflection

**a) What surprised us, and how did we check it?** The approximately 18% increase in average daily orders stood out. We recalculated unique transactions for both periods and checked for exact duplicate rows, missing values, and transaction IDs associated with more than one store or date. Those checks did not reveal obvious transaction-assignment errors.

**b) Which result are we least confident in?** The reported rainfall correlation of **0.445** is the least certain interpretation. It is based on only **30 days** and weather observations from Charlottesville, rather than weather matched to each Florida store. The join itself can be valid even when the geographic match is not.

**c) What would we improve next?** We would verify the actual hurricane event window, obtain store-level geographic metadata, retrieve local weather for each store, and test whether the demand increase persists under alternative comparison windows. We would also document and test all product and store normalization rules.

*These reflections build on the team's October 30 progress notebook and should be reviewed by all team members before submission.*